# 06 · Swap in a different cost model

The navigation & search cost (paper §3.4.1) gives each simulated step a fixed cost. The framework is designed for richer cost models (§5.3). To use one, subclass `CostModel`, or `DefaultCostModel` to change only some steps, and pass it to `evaluate`. Hits, adjustments and misses are still determined the same way. Only the prices change.

Suppose moving between facets inside an app is slow on your device, so each hop of a same-app adjustment costs 3 instead of 1. Suggesting the *right app but the wrong facet* then pays off less, and exact facet hits matter more.

In [ ]:
from contextxr.metrics.cost_model import DefaultCostModel


class CostlyAdjustment(DefaultCostModel):
    name = "costly_adjustment"

    def adjust(self, target, suggested, graph):
        hops = graph.hops(suggested.facet_id, target.facet_id)
        return None if hops is None else 3.0 * hops

In [ ]:
from contextxr import evaluate, load_minexrpp
from contextxr.methods.registry import build_method

ds = load_minexrpp()
names = ["popularity", "relational", "zero_shot_llm", "few_shot_llm[environment]", "few_shot_llm[user]", "few_shot_llm[user+env]"]
rows = []
for name in names:
    method, protocol = build_method(name, task="initial", dataset=ds)
    default = evaluate(method, "initial", ds, **protocol).summary
    costly = evaluate(method, "initial", ds, cost_model=CostlyAdjustment(), **protocol).summary
    rows.append((name, default["cost_reduction_pct"], costly["cost_reduction_pct"]))

In [ ]:
from IPython.display import Markdown

rank = lambda col: {name: i + 1 for i, (name, *_) in enumerate(sorted(rows, key=lambda r: r[col]))}
r1, r2 = rank(1), rank(2)
lines = ["| method | reduction (default) | rank | reduction (costly adjustment) | rank |", "|---|---:|---:|---:|---:|"]
lines += [f"| {n} | {a:.1f}% | {r1[n]} | {b:.1f}% | {r2[n]} |" for n, a, b in rows]
Markdown("\n".join(lines))

| method | reduction (default) | rank | reduction (costly adjustment) | rank |
|---|---:|---:|---:|---:|
| popularity | -6.4% | 6 | -1.5% | 6 |
| relational | -13.3% | 5 | -9.6% | 4 |
| zero_shot_llm | -16.7% | 4 | -8.4% | 5 |
| few_shot_llm[environment] | -18.8% | 3 | -10.7% | 3 |
| few_shot_llm[user] | -22.6% | 2 | -16.7% | 1 |
| few_shot_llm[user+env] | -22.7% | 1 | -15.6% | 2 |

Every method gains less once adjustments are expensive. The ranking also changes. Relational retrieval moves ahead of the zero-shot LLM, which often picks the right app at the wrong facet granularity (Fig. 5). User-matched examples move ahead of User+Env. Report the cost model alongside your results. Every results file records it under `cost_model`.